# Pipeline `Upgrade` — PIGuard + Novo MOF + Laya English (Execução Única no Google Colab)

Este notebook executa integralmente o pipeline **`Upgrade`** em **um único ambiente Google Colab**, sem dependência de `JevTest` e obedecendo estritamente às seguintes premissas metodológicas:

1. **Preservação Integral das Bases**: Nenhuma amostra é removida de `train_cleaned_original.json` (`75.538`) nem de `train_augmented_corrected.json` (`75.973` exemplos intactos).
2. **Independência dos Artefatos MOF Antigos e Descarte de `valid.json` para Calibração**: Os arquivos antigos `MOF.json` e `MOF.candidates.json` **não** são utilizados, e `datasets/valid.json` é descartado para calibração por possuir sobreposição com `NotInject`, `BIPIA` e `WildGuard`.
3. **Ordem de Execução**:
   - **Etapa 0 (Início)**: Geração do pool do **Novo MOF** a partir dos `253` gatilhos de `Corrections/augmented_corrected/trigger_words.report.json`, particionado de forma 100% disjunta em **`MOF_v2_train.json` (`762` amostras)** e **`calibration_heldout.json` (`238` excedentes benignos + `102` ataques extras pelo funil de 4 filtros)**.
   - **Tópico 1**: Avaliação dos **5 modelos `rp`** salvos em `RESULTADOS_SEED42_MAX512` (sem retreiná-los), comparando `PIGuard Puro` vs. `+ Laya English (Não Calibrado)` vs. `+ Laya English (Calibrado)`.
   - **Tópico 2**: Retreinamento *scratch* de **`train_augmented_corrected_new_mof`** (`75.973` intactos + `762` Novo MOF = `76.735` exemplos, `seed=42`, `max_length=512`) e avaliação `Puro` vs. `+ Laya English`.
   - **Tópico 3**: Consolidação geral de tabelas, intervalos de confiança (IC 95%), latência e gráficos.


## 1. Ambiente, Google Drive, Dependências e Notificações (`ntfy.sh`)

Esta célula monta o Google Drive em `/content/drive`, sincroniza o repositório `/content/RP_PIGuard`, instala as dependências necessárias (incluindo `transformers>=4.48.0,<5.0.0` para suporte nativo ao backbone `ModernBERT-large` do **Laya English**, além de `safetensors` e `huggingface_hub`), carrega `OPENROUTER_API_KEY` dos Secrets do Colab e configura notificações via `ntfy.sh` (`piguard_gabriel_treinos`).


In [ ]:
import os
import sys
import json
import getpass
import subprocess
import time
import traceback
import urllib.request
from collections import deque
from contextlib import contextmanager
from pathlib import Path

NTFY_TOPIC = "piguard_gabriel_treinos"
NTFY_URL = "https://ntfy.sh/"
NTFY_ENABLED = True

def notify_ntfy(title, message, *, priority=3, tags=None, topic=NTFY_TOPIC, timeout=15):
    if not NTFY_ENABLED:
        return
    payload = {
        "topic": topic,
        "title": str(title)[:250],
        "message": str(message)[:3800],
        "priority": int(priority),
    }
    if tags:
        payload["tags"] = list(tags)
    data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    req = urllib.request.Request(
        NTFY_URL,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            resp.read()
    except Exception as exc:
        print(f"[ntfy] Aviso ao enviar notificação ({title}): {exc}", flush=True)

@contextmanager
def ntfy_cell(cell_name):
    started = time.monotonic()
    notes = []
    try:
        yield notes
    except BaseException as exc:
        elapsed = time.monotonic() - started
        tb_tail = "".join(traceback.format_exception_only(type(exc), exc)).strip()
        msg = f"Célula '{cell_name}' falhou após {elapsed:.1f}s.\nErro: {tb_tail}"
        if notes:
            msg += "\nContexto:\n" + "\n".join(str(n) for n in notes)
        notify_ntfy(
            title=f"[PIGuard Upgrade] Erro: {cell_name}",
            message=msg,
            priority=4,
            tags=["x", "warning"],
        )
        raise
    else:
        elapsed = time.monotonic() - started
        msg = f"Célula '{cell_name}' concluída com sucesso em {elapsed:.1f}s."
        if notes:
            msg += "\n" + "\n".join(str(n) for n in notes)
        notify_ntfy(
            title=f"[PIGuard Upgrade] Concluída: {cell_name}",
            message=msg,
            priority=3,
            tags=["white_check_mark"],
        )

def run_cmd(command, *, cwd=None):
    proc = subprocess.run(command, cwd=str(cwd) if cwd else None, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"Comando falhou ({proc.returncode}): {' '.join(map(str, command))}")

with ntfy_cell("1. Ambiente e Caminhos") as ntfy_notes:
    IS_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
    if IS_COLAB:
        from google.colab import drive, userdata
        drive.mount("/content/drive")

        REPO = Path("/content/RP_PIGuard")
        if not REPO.exists():
            token = None
            try:
                token = (userdata.get("GITHUB_TOKEN") or "").strip() or None
            except Exception:
                token = None
            repo_url = (
                f"https://{token}@github.com/gabrielbelo2007/RP_PIGuard.git"
                if token else "https://github.com/gabrielbelo2007/RP_PIGuard.git"
            )
            run_cmd(["git", "clone", repo_url, str(REPO)])
        elif (REPO / ".git").exists():
            pull = subprocess.run(
                ["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"],
                capture_output=True, text=True,
            )
            if pull.returncode != 0:
                print(f"Aviso git pull: {pull.stderr.strip() or pull.stdout.strip()}")

        run_cmd([
            sys.executable, "-m", "pip", "install", "-q",
            "transformers>=4.48.0,<5.0.0", "datasets>=2.21.0",
            "safetensors>=0.4.3", "huggingface_hub>=0.24.0",
            "sentencepiece", "protobuf", "tiktoken", "pyyaml",
            "tqdm", "accelerate", "pandas", "matplotlib",
        ])
    else:
        REPO = next(
            (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "PIGuard.py").is_file()),
            Path.cwd(),
        )

    os.chdir(REPO)
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))

    import torch
    import transformers

    DRIVE_ROOT = Path("/content/drive/MyDrive/RP_PIGuard_outputs") if IS_COLAB else (REPO / "no_git" / "drive_mock")
    CORRECTIONS_DIR = DRIVE_ROOT / "Corrections"
    DRIVE_RP_ROOT = DRIVE_ROOT / "RESULTADOS_SEED42_MAX512"
    UPGRADE_OUTPUT = (DRIVE_ROOT / "Upgrade") if IS_COLAB else (REPO / "Upgrade" / "outputs")
    UPGRADE_OUTPUT.mkdir(parents=True, exist_ok=True)

    # Configurações principais do experimento
    SEED = 42
    FPR_BUDGET = 0.01       # +1.0 pp máximo de FPR adicional no held-out benigno durante a calibração
    N_BOOT = 2000           # Réplicas do Paired Bootstrap (IC 95%)
    FORCE_STEP0 = False
    FORCE_TOPIC1 = False
    FORCE_RETRAIN_TOPIC2 = False
    FORCE_EVAL_TOPIC2 = False

    def ensure_api_key(*, prompt_if_missing=False):
        key = None
        if IS_COLAB:
            from google.colab import userdata
            for sname in ("OPENROUTER_API_KEY", "OPENAI_API_KEY"):
                try:
                    val = userdata.get(sname)
                    if val and val.strip():
                        key = val.strip().strip("\"'")
                        break
                except Exception:
                    pass
        if not key:
            for env_var in ("OPENROUTER_API_KEY", "OPENAI_API_KEY"):
                val = os.environ.get(env_var)
                if val and val.strip():
                    key = val.strip().strip("\"'")
                    break
        if not key:
            for env_path in (REPO / ".env", DRIVE_ROOT / ".env", Path("/content/.env")):
                if env_path.is_file():
                    for line in env_path.read_text(encoding="utf-8").splitlines():
                        if "=" in line and not line.strip().startswith("#"):
                            k, v = line.split("=", 1)
                            if k.strip().lower() in ("openrouter_api_key", "openai_api_key") and v.strip():
                                key = v.strip().strip("\"'")
                                break
                if key:
                    break
        if not key and prompt_if_missing:
            key = getpass.getpass("Cole sua OPENROUTER_API_KEY: ").strip().strip("\"'")
        if key:
            os.environ["OPENROUTER_API_KEY"] = key
        return key

    api_present = bool(ensure_api_key(prompt_if_missing=False))
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    print({
        "repo": str(REPO),
        "drive_rp_root": str(DRIVE_RP_ROOT),
        "corrections_dir": str(CORRECTIONS_DIR),
        "upgrade_output": str(UPGRADE_OUTPUT),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "gpu": gpu_name,
        "api_key_present": api_present,
    })
    ntfy_notes.append(f"gpu={gpu_name} | transformers={transformers.__version__} | output={UPGRADE_OUTPUT}")


## 2. Etapa 0 (Início): Pool do Novo MOF + Separação dos Excedentes *Held-Out*

### Origem das Amostras *Held-Out* e Garantias de Integridade
- **Nenhuma amostra** é retirada de `train_cleaned_original.json` (`75.538`) nem de `train_augmented_corrected.json` (`75.973` permanecem 100% intactos).
- **Nenhum artefato antigo** (`MOF.json`, `MOF.candidates.json` ou `valid.json`) é utilizado.
- A partir dos **`253` gatilhos** de `Corrections/augmented_corrected/trigger_words.report.json`, sorteamos combinações de $k \in \{1, 2, 3\}$ gatilhos distintos com amostragem ponderada sem reposição ($20\%$ uniforme + $80\%$ proporcional à severidade $s(w) = \max(p_{\text{inj}}(w) - 0.50, 0)$) distribuída nos **4 domínios críticos** (*Código/SQL/JSON/CLI*, *Documentos/E-mails/Tabelas*, *Escrita Criativa/Roleplay*, *Consultas Gerais/Acadêmicas*), com limite estrito de $\le 256$ tokens no tokenizador `microsoft/deberta-v3-base`, aprovação `NO` do juiz LLM e filtro de similaridade + disjunção total contra `valid.json`, benchmarks e dados de treino.
- Após pontuar o pool aprovado com `Corrections/augmented_corrected/best_model.pth`, realizamos um particionamento 100% disjunto:
  - **`MOF_v2_train.json` (`762` exemplos benignos)**: cotas exatas de `152` ($k=1$, 20%), `229` ($k=2$, 30%) e `381` ($k=3$, 50%), selecionadas via **Seleção Contextual *Hard-Negative* (80% maiores scores $P_{\text{PI}}(\text{inj}\mid x)$ + 20% aleatórios)**;
  - **`calibration_heldout.json` (`340` exemplos *held-out*)**: **`238` excedentes benignos** do mesmo pool estratificados por domínio e quantil de score (`48` de $k=1$, `71` de $k=2$, `119` de $k=3$) + **`102` ataques validados** pelo funil de 4 filtros (`6` por cada um dos `17` formatos, fora dos `435` exemplos de `LLM_augmented.json`).


In [ ]:
with ntfy_cell("2. Etapa 0: Pool do Novo MOF + Held-Out") as ntfy_notes:
    from Upgrade.step0_new_mof_and_heldout import run_step0

    ensure_api_key(prompt_if_missing=True)
    trigger_report_path = CORRECTIONS_DIR / "augmented_corrected" / "trigger_words.report.json"
    if not trigger_report_path.is_file():
        trigger_report_path = CORRECTIONS_DIR / "augmented_corrected" / "trigger_words.json"
    corrected_ckpt = CORRECTIONS_DIR / "augmented_corrected" / "best_model.pth"

    step0_dir = UPGRADE_OUTPUT / "step0"
    step0_report = run_step0(
        trigger_path=trigger_report_path,
        checkpoint_path=corrected_ckpt,
        output_dir=step0_dir,
        repo_root=REPO,
        corrections_dir=CORRECTIONS_DIR,
        workers=5,
        seed=SEED,
        force=FORCE_STEP0,
        api_key=os.environ.get("OPENROUTER_API_KEY"),
    )
    MOF_V2_PATH = Path(step0_report["mof_v2_train_path"])
    HELDOUT_PATH = Path(step0_report["calibration_heldout_path"])

    print("MOF_v2_train:", step0_report["mof_v2_count"], "| k:", step0_report["mof_v2_k_distribution"])
    print("Domínios MOF_v2:", step0_report["mof_v2_domain_distribution"])
    print("Seleção MOF_v2:", step0_report["mof_v2_selection_modes"])
    print("Held-Out total:", step0_report["heldout_total_count"],
          "| benign:", step0_report["heldout_benign_count"],
          "| malicious:", step0_report["heldout_malicious_count"])
    print("Disjunção total:", step0_report["disjoint_check"])
    assert step0_report["disjoint_check"]["disjoint"], "Vazamento detectado na Etapa 0!"
    ntfy_notes.append(
        f"MOF_v2={step0_report['mof_v2_count']} | HeldOut={step0_report['heldout_total_count']} | Disjunto=True"
    )


## 3. Fundamentação Matemática do Laya English, Agregação, Limiar $T_L$, Calibração e *Paired Bootstrap*

### 3.1 Laya Versão English (`convaiinnovations/laya`)
Carregamos diretamente no mesmo processo Python o modelo **Laya English** (`answerdotai/ModernBERT-large` + cabeça Transformer de 2 camadas com marcadores `[MASK]`, contexto nativo até 8.192 tokens).

### 3.2 Função de Agregação em Dois Níveis
Quando o PIGuard classifica uma entrada $x$ como **Benigna ($\hat{y}_{\text{PI}}(x) = 0$)**, o Laya English é ativado (se $\hat{y}_{\text{PI}}(x) = 1$, o Laya **não** é acionado — preservando 100% dos Verdadeiros Positivos do PIGuard a `0 ms` de custo extra):
1. **Nível 1 — Agregação Intra-Janela**: Para uma janela $s_j$ com opções $\mathcal{C} = \{\text{benign}\} \cup \mathcal{C}_{\text{inj}}$ nos marcadores `[MASK]`, aplicamos a temperatura $T$:
   $$p_c(s_j; T) = \frac{\exp(z_c(s_j)/T)}{\sum_{c' \in \mathcal{C}} \exp(z_{c'}(s_j)/T)}, \quad P_{\text{Laya}}(\text{inj} \mid s_j; T) = \sum_{c \in \mathcal{C}_{\text{inj}}} p_c(s_j; T) = 1 - p_{\text{benign}}(s_j; T)$$
2. **Nível 2 — Agregação Inter-Janelas ($\max$ com Sobreposição de 512 Tokens)**: Textos acima do orçamento por passagem são fatiados em janelas deslizantes $s_1, \dots, s_M$ com **512 tokens de sobreposição**, agregadas pelo operador máximo:
   $$S_{\text{Laya}}(x; T) = \max_{j \in \{1, \dots, M\}} P_{\text{Laya}}(\text{inj} \mid s_j; T)$$

### 3.3 Limiar $T_L$ e Comparação Não Calibrado vs. Calibrado
- **$T_L \in (0, 1)$** é o **Limiar de Mudança de Decisão**: quando o PIGuard prevê `0`, a cascata inverte a predição para `1` se e somente se $S_{\text{Laya}}(x; T) > T_L$.
- **Laya Não Calibrado**: usa $T = 1.0$ (logits brutos) e $T_L = 0.50$.
- **Laya Calibrado**: ajusta $T^*$ minimizando a *Negative Log-Likelihood* em `calibration_heldout.json` e escolhe o menor $T_L^* \in [0.50, 0.99]$ tal que o aumento de Falsos Positivos nos `238` benignos *held-out* satisfaça $\text{FPR}_{\text{casc}}(T_L^*; T^*) - \text{FPR}_{\text{PI}} \le \epsilon_{\text{FPR}}$ ($\epsilon_{\text{FPR}} = 1.0\text{ pp}$).

### 3.4 *Paired Bootstrap* (IC 95%)
Em $B = 2.000$ réplicas (`seed=42`), sorteamos com reposição os **mesmos índices** $I_k^{(b)}$ para ambos os modelos (`base` e `cascata`), isolando exclusivamente a variância das mudanças de decisão do Laya para estimar o $\text{IC}_{95\%}(\Delta)$.

A célula abaixo instancia o Laya English e pré-extrai (com cache em disco) os logits por janela para o conjunto *held-out* e para todos os benchmarks.


In [ ]:
with ntfy_cell("3. Carregamento e Pré-Extração Laya English") as ntfy_notes:
    from Upgrade import load_benchmarks, read_json
    from Upgrade.laya_english_cascade import LayaEnglishEngine
    from Upgrade.topic1_rp_laya_evaluation import extract_or_load_laya_cache

    CACHE_DIR = UPGRADE_OUTPUT / "cache"
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    HELDOUT_PATH = globals().get("HELDOUT_PATH", UPGRADE_OUTPUT / "step0" / "calibration_heldout.json")

    benchmarks = load_benchmarks(REPO / "datasets", include_tensortrust=True)
    heldout_records = read_json(HELDOUT_PATH)

    bench_cache_file = CACHE_DIR / "laya_english_benchmark_logits.json"
    held_cache_file = CACHE_DIR / "laya_english_heldout_logits.json"

    laya_engine = None
    if not (bench_cache_file.is_file() and held_cache_file.is_file()):
        laya_engine = LayaEnglishEngine(window_max_len=2048, overlap_tokens=512)

    bench_laya_cache = extract_or_load_laya_cache(
        {k: texts for k, (texts, _) in benchmarks.items()},
        bench_cache_file,
        laya_engine=laya_engine,
        batch_size=16,
    )
    heldout_laya_cache = extract_or_load_laya_cache(
        {"heldout": [str(r["prompt"]) for r in heldout_records]},
        held_cache_file,
        laya_engine=laya_engine,
        batch_size=16,
    )
    if laya_engine is not None:
        del laya_engine
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("Logits Laya English prontos para", list(bench_laya_cache["entries"].keys()))
    ntfy_notes.append(f"Laya English pré-extraído para {len(bench_laya_cache['entries'])} benchmarks + heldout")


## 4. Tópico 1: Avaliação dos 5 Modelos `rp` de `RESULTADOS_SEED42_MAX512` com Laya English

Reutiliza diretamente os 5 checkpoints `.pth` marcados como `rp` (`train_augmented_rp`, `train_clean_mof_rp`, `train_augmented_mof_finetuning_rp`, `train_augmented_mof_rp`, `train_augmented_shortcut_rp`) sem retreiná-los, comparando:
1. **PIGuard Puro** (comparado contra o cache original salvo em `RESULTADOS_SEED42_MAX512/Gráficos/_cache/`);
2. **PIGuard + Laya English Não Calibrado** ($T = 1.0, T_L = 0.50$);
3. **PIGuard + Laya English Calibrado** ($T^*, T_L^*$ ajustados em `calibration_heldout.json`).


In [ ]:
with ntfy_cell("4. Tópico 1: 5 Modelos rp + Laya English") as ntfy_notes:
    import pandas as pd
    from IPython.display import display
    from Upgrade.topic1_rp_laya_evaluation import RP_MODELS, run_topic1

    HELDOUT_PATH = globals().get("HELDOUT_PATH", UPGRADE_OUTPUT / "step0" / "calibration_heldout.json")

    t1_payload = run_topic1(
        drive_rp_root=DRIVE_RP_ROOT,
        heldout_path=HELDOUT_PATH,
        output_dir=UPGRADE_OUTPUT,
        repo_root=REPO,
        models=RP_MODELS,
        fpr_budget=FPR_BUDGET,
        n_boot=N_BOOT,
        seed=SEED,
        force=FORCE_TOPIC1,
    )
    t1_df = pd.DataFrame(t1_payload["summary_rows"])
    display(t1_df[[
        "model", "mode_label", "temperature", "threshold_tl",
        "overdefense", "benign", "malicious", "average",
        "delta_average_pp", "ci95_average_low", "ci95_average_high",
        "call_rate_pct", "inference_ms", "p50_ms", "p95_ms", "efficiency"
    ]].round(3))

    for res in t1_payload["results"]:
        cal = res["calibration"]
        avg_cal = res["modes"]["laya_calibrated"]["metrics"]["average"]
        avg_pure = res["modes"]["pure"]["metrics"]["average"]
        ntfy_notes.append(
            f"{res['model']}: {avg_pure:.2f}% -> {avg_cal:.2f}% (T*={cal['temperature_star']:.2f}, TL*={cal['threshold_tl']:.3f})"
        )


## 5. Tópico 2: Retreinamento *Scratch* com `train_augmented_corrected` Intacto + Novo MOF (`762`)

- Combina **`Corrections/data/train_augmented_corrected.json` (`75.973` exemplos 100% intactos)** com **`MOF_v2_train.json` (`762` exemplos)** = **`76.735` exemplos** (mesmo volume exato de `datasets/train.json`).
- Treina `train_augmented_corrected_new_mof` do zero (`microsoft/deberta-v3-base`, `seed=42`, `max_length=512`, `epochs=3`, `batch_size=32`, `lr=2e-5`, `warmup=100`, `save_step=200`).
- Avalia nas 3 modalidades (`Puro`, `+ Laya English Não Calibrado`, `+ Laya English Calibrado`).


In [ ]:
with ntfy_cell("5. Tópico 2: Treino e Avaliação train_augmented_corrected_new_mof") as ntfy_notes:
    import pandas as pd
    from IPython.display import display
    from Upgrade.topic2_corrected_new_mof import run_topic2

    MOF_V2_PATH = globals().get("MOF_V2_PATH", UPGRADE_OUTPUT / "step0" / "MOF_v2_train.json")
    HELDOUT_PATH = globals().get("HELDOUT_PATH", UPGRADE_OUTPUT / "step0" / "calibration_heldout.json")

    corrected_train_file = CORRECTIONS_DIR / "data" / "train_augmented_corrected.json"
    llm_aug_file = CORRECTIONS_DIR / "augmented" / "LLM_augmented.json"

    t2_payload = run_topic2(
        mof_v2_path=MOF_V2_PATH,
        heldout_path=HELDOUT_PATH,
        output_dir=UPGRADE_OUTPUT,
        corrected_train_path=corrected_train_file,
        llm_augmented_path=llm_aug_file,
        repo_root=REPO,
        fpr_budget=FPR_BUDGET,
        n_boot=N_BOOT,
        seed=SEED,
        force_retrain=FORCE_RETRAIN_TOPIC2,
        force_eval=FORCE_EVAL_TOPIC2,
    )
    print("Dataset Tópico 2:", t2_payload["dataset_stats"])
    t2_df = pd.DataFrame(t2_payload["summary_rows"])
    display(t2_df[[
        "model", "mode_label", "temperature", "threshold_tl",
        "overdefense", "benign", "malicious", "average",
        "delta_average_pp", "ci95_average_low", "ci95_average_high",
        "call_rate_pct", "inference_ms", "p50_ms", "p95_ms", "efficiency"
    ]].round(3))

    for row in t2_payload["summary_rows"]:
        ntfy_notes.append(f"{row['mode_label']}: avg={row['average']:.2f}% | mal={row['malicious']:.2f}% | over={row['overdefense']:.2f}%")


## 6. Tópico 3: Consolidação Geral (Tabelas, ICs, Latência e Gráficos)

Unifica os resultados dos Tópicos 1 e 2 com os referenciais oficiais de `RESULTADOS_SEED42_MAX512` (`PIGuard HF`, `train_augmented_mof_original`) e de `Corrections` (`augmented_corrected`, `augmented_mof_corrected`), salvando todas as tabelas CSV, o relatório Markdown completo e os gráficos comparativos em `Upgrade/topic3/`.


In [ ]:
with ntfy_cell("6. Tópico 3: Consolidação Geral") as ntfy_notes:
    import pandas as pd
    from IPython.display import Image, display
    from Upgrade.topic3_consolidated_report import run_topic3

    t3_bundle = run_topic3(
        output_dir=UPGRADE_OUTPUT,
        drive_rp_root=DRIVE_RP_ROOT,
        corrections_dir=CORRECTIONS_DIR,
    )

    print("=== Evolução do MOF na Base Corrigida (Tópico 2) ===")
    display(pd.DataFrame(t3_bundle["topic2_rows"])[[
        "display_name", "overdefense", "benign", "malicious", "average", "inference_ms", "mof_note"
    ]].round(2))

    print("=== Consolidação Mestra (Tópico 3) ===")
    display(pd.DataFrame(t3_bundle["master_rows"])[[
        "display_name", "overdefense", "benign", "malicious", "average", "call_rate_pct", "inference_ms", "mof_note"
    ]].round(2))

    for chart_name, chart_path in t3_bundle["charts"].items():
        print(f"Gráfico [{chart_name}]: {chart_path}")
        display(Image(filename=chart_path))

    ntfy_notes.append(f"Relatório consolidado: {t3_bundle['markdown_report']}")
